In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import scienceplots
plt.style.use(['science', 'nature', 'grid'])

import numpy as np
from scipy.interpolate import interp1d
import pandas as pd

from LoadMoments import *
from LoadTSV import *

In [ ]:
# Plotting
colors = ['blue', 'orange', 'green', 'red', 'purple', 'brown']
linestyles = ['-', '--', ':', '-.', (0, (3, 1, 1, 1)), (0, (5, 1))]

# Parameter
M = 4
closure = "ExtGram"
Kn = 1.0
source_string = "relaxation_source"
T_end = 0.3
base_tree_level = 10
polydeg = 1
rho_L = 7.0
v_L1 = 0.0
v_L2 = 0.0
v_L3 = 0.0
theta_L = 1.0
rho_R = 1.0
v_R1 = 0.0
v_R2 = 0.0
v_R3 = 0.0
theta_R = 1.0

# Solution files
n_angle_pairs = 4

marker = ['o', 's', '^', 'x', 'd']

# Kn and relaxation for solution files
Kn = 1.0
source = 'relaxation_source'
name = lambda anglepairnumber, Kn, source_string: f"../out/1D3D/1D3D_angles/gram_solution_M{M}_closure{closure}_Kn{Kn}_source{source_string}_T_end{T_end}_rho_L{rho_L}_rho_R{rho_R}_v_L1{v_L1}_v_L2{v_L2}_v_L3{v_L3}_v_R1{v_R1}_v_R2{v_R2}_v_R3{v_R3}_theta_L{theta_L}_theta_R{theta_R}_base_tree_level{base_tree_level}_polydeg{polydeg}_anglepairnumber{anglepairnumber}_cons.tsv"
name_reduced = lambda anglepair, Kn, source_string: f"../out/1D3V/1D3V_angles/gram_solution_M{M}_closure{closure}_Kn{Kn}_source{source_string}_anglepair{anglepair}_cons.tsv"

In [ ]:
angle_pairs = ['Max', 'Arc', 'Det']
# Stable angles
for i, angle_pair in enumerate(angle_pairs):

    fig, axs = plt.subplots(figsize=(6.0 * 3/4, 2.5))
    twx = axs.twinx()
    twx.grid(False)

    axs.set_xlim(-1, 1)
    axs.set_xlabel('x')
    axs.set_ylabel('$\\rho, p$')
    twx.set_ylabel('$v$')

    # legacy 1D-3D slab
    solution_file = name(i, Kn, source)
    data = read_solution_file(solution_file)
    
    times = np.array([data[ts]["t"] for ts in sorted(data.keys())])
    timesteps = np.array(sorted(data.keys()))
    
    time_query = 21 
    sol = interpolate_solution(data, time_query, times, timesteps)
    x, U = sol["x"], sol["u"]

    rho = U[:, 0]
    v = U[:, 1] / U[:, 0]
    theta = 1 / (3 * rho) * (U[:, 2] + 2 * U[:, 3] - rho * v**2) 
    p = rho * theta

    axs.plot(x, rho, color=colors[0], linestyle=linestyles[0])
    twx.plot(x, v, color=colors[1], linestyle=linestyles[0])
    axs.plot(x, p, color=colors[2], linestyle=linestyles[0])

    # Reduced 1D-3V slab
    solution_file = name_reduced(angle_pair, Kn, source)
    data = read_solution_file(solution_file)
    
    times = np.array([data[ts]["t"] for ts in sorted(data.keys())])
    timesteps = np.array(sorted(data.keys()))
    
    time_query = 21 
    sol = interpolate_solution(data, time_query, times, timesteps)
    x, U = sol["x"], sol["u"]

    rho = U[:, 0]
    v = U[:, 1] / U[:, 0]
    theta = 1 / (3 * rho) * (U[:, 2] + 2 * U[:, 3] - rho * v**2) 
    p = rho * theta

    axs.plot(x, rho, color=colors[0], linestyle=linestyles[1])
    twx.plot(x, v, color=colors[1], linestyle=linestyles[1])
    axs.plot(x, p, color=colors[2], linestyle=linestyles[1])

    # Legend and Layout
    axs.plot([], [], color=colors[0], lw=2, label='$\\rho$')
    axs.plot([], [], color=colors[1], lw=2, label='$v$')
    axs.plot([], [], color=colors[2], lw=2, label='$p$')

    axs.plot([], [], color='black', linestyle=linestyles[0], lw=1, label='Old Implementation')
    axs.plot([], [], color='black', linestyle=linestyles[1], lw=1, label='New Implementation')

    axs.set_title(f'Angle Pair: {angle_pair}')
    axs.legend(bbox_to_anchor=(1.3, 1.0), loc='upper center')

    plt.show()
    fig.savefig(f'../out/Figures/1D3V/slab/ModelComparison_anglepair{angle_pair}.pdf')

In [ ]:
M_vec = [4, 6, 8]

for angle_pair in angle_pairs:
    fig, axs = plt.subplots(1, 3, figsize=(12, 3))
    fig.suptitle(f'Angle Pair: {angle_pair}', fontsize=16)

    [ax.set_xlim(-1, 1) for ax in axs]
    [ax.set_xlabel('x') for ax in axs]
    [ax.set_ylabel('$\\rho, p$') for ax in axs]
    for i, M in enumerate(M_vec):
        for j, (Kn, source) in enumerate([(0.1, 'relaxation_source'), (1.0, 'relaxation_source'), (10.0, 'zero_source')]):
            ax = axs[j]
            twx = ax.twinx()
            twx.grid(False)
            twx.set_ylabel('$v$')

            ## Reduced 1D-3V slab
            solution_file = name_reduced(angle_pair, Kn, source)
            data = read_solution_file(solution_file)
            
            times = np.array([data[ts]["t"] for ts in sorted(data.keys())])
            timesteps = np.array(sorted(data.keys()))
            
            time_query = 21 
            sol = interpolate_solution(data, time_query, times, timesteps)
            x, U = sol["x"], sol["u"]

            rho = U[:, 0]
            v = U[:, 1] / U[:, 0]
            theta = 1 / (3 * rho) * (U[:, 2] + 2 * U[:, 3] - rho * v**2) 
            p = rho * theta

            ax.plot(x, rho, color=colors[0], linestyle=linestyles[i])
            twx.plot(x, v, color=colors[1], linestyle=linestyles[i])
            ax.plot(x, p, color=colors[2], linestyle=linestyles[i])

            ax.plot([], [], color='black', linestyle=linestyles[i], lw=1, label=f'M={M}')

            ax.legend(bbox_to_anchor=(1.3, 1.0), loc='upper center')
            if source=='relaxation_source':
                ax.set_title(f'Kn={Kn}', fontsize=12)
            else:
                ax.set_title('Kn$\\to \\infty$', fontsize=12)

    fig.tight_layout(); fig.show()
    fig.savefig(f'../out/Figures/1D3V/slab/ModelConvergence_closure{closure}_anglepair{angle_pair}.pdf')